In [8]:
# -*- coding: utf-8 -*-
"""s1_dynamic_bertscore.py

Reference-based BERTScore (vs gold) for the dynamic few-shot outputs of
healthslm_eval_dynamic_fewshot.py — one run per retriever
(S-PubMedBERT / BGE / TF-IDF). BERTScore ONLY (no ROUGE), so it can be
run/rerun independently of s1_dynamic_reference_quality.py.

Scoring identical to s1_reference_quality.py / s1_dynamic_reference_quality.py:
  - BERTScore distilbert-base-uncased, RAW (rescale_with_baseline=False)

Input: per-example CSVs written by healthslm_eval_dynamic_fewshot.py in
/workspace/results, named results_dynamic_k5*{model}*{retriever}*per_example*.csv
  columns: dataset, task, k_shot, reference, bounds, raw_prediction,
           extracted, retrieval_sim
  'extracted' = FULL generation text; 'dataset' values are test FILENAMES.
  Rows aligned positionally across retrievers; verified via reference equality.
  instance_idx = row position within task (same convention as
  s3_dynamic_nli_consistency.py / s1_dynamic_reference_quality.py).

Outputs (in OUT_DIR):
  s1dynbs_{MODEL_SLUG}_per_output.csv   — task x retriever x instance_idx
  s1dynbs_{MODEL_SLUG}_per_instance.csv — task x instance_idx, wide:
        {retr}_bertscore per retriever + retr_mean/min/max_bertscore,
        retr_bertscore_spread, n_valid_retr_outputs
  s1dynbs_{MODEL_SLUG}_summary.csv      — task x retriever

Run:
    pip install bert-score pandas torch
    python s1_dynamic_bertscore.py
"""

import os
import glob as _glob
import numpy as np
import pandas as pd
import torch

# ------------------------------------------------------------------------
# Config (matches s1_dynamic_reference_quality.py / s3_dynamic_nli_consistency.py)
# ------------------------------------------------------------------------

MODEL_NAME = "medalpaca/medalpaca-7b"   # <- must match the eval runs
MODEL_SLUG = MODEL_NAME.split('/')[-1].lower().replace('-', '_').replace('.', '_')

OUT_DIR = '/workspace/demo_sensitivity_runs'
RESULTS_DIR = '/workspace/results'


def _find_per_example(retriever_pat):
    pats = [f'{RESULTS_DIR}/results_dynamic_k5*{MODEL_SLUG}*{retriever_pat}*per_example*.csv']
    hits = sorted(set(sum((_glob.glob(p) for p in pats), [])))
    if len(hits) != 1:
        raise FileNotFoundError(
            f"expected exactly 1 per-example CSV for '{retriever_pat}', "
            f"found {len(hits)}: {hits}")
    return hits[0]

RETRIEVER_CSVS = {
    's_pubmedbert': _find_per_example('s_pubmedbert'),
    'bge':          _find_per_example('bge'),
    'tfidf':        _find_per_example('tfidf'),
}
print("Per-example CSVs:")
for r, p in RETRIEVER_CSVS.items():
    print(f"  {r}: {os.path.basename(p)}")

# Canonical task label -> 'dataset' value (filenames). aci excluded per design.
TASK_MAP = {
    'meddialog':      'meddialog_test_sample.jsonl',
    'medicationqa':   'MedicationQA_test_sample.jsonl',
    'mtsamples':      'mtsamples_test_sample.jsonl',
    'mtsamples_proc': 'mtsamples_procedures_test_sample.jsonl',
}
TASKS = list(TASK_MAP)

TEXT_COL = 'extracted'
REF_COL = 'reference'
TASK_COL = 'dataset'
RETRIEVERS = list(RETRIEVER_CSVS)

# Reference-based BERTScore: fixed checkpoint, RAW (matches old eval sheet)
BERT_MODEL = 'distilbert-base-uncased'
BERT_RESCALE = False
BERT_BATCH = 64
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# ------------------------------------------------------------------------
# Load + align (same guards as the other dynamic scripts)
# ------------------------------------------------------------------------

def load_retriever_outputs():
    """{task: (refs, {retriever: [output_text, ...]}, keep_idxs)} positionally aligned."""
    frames = {}
    for r, path in RETRIEVER_CSVS.items():
        df = pd.read_csv(path, keep_default_na=False, dtype=str)
        assert TEXT_COL in df.columns, f"{path}: no '{TEXT_COL}' column"
        frames[r] = df

    per_task = {}
    for task in TASKS:
        subs = {r: f[f[TASK_COL] == TASK_MAP[task]].reset_index(drop=True)
                for r, f in frames.items()}
        ns = {r: len(s) for r, s in subs.items()}
        if len(set(ns.values())) != 1 or 0 in ns.values():
            print(f"  [skip] {task}: row counts differ or empty: {ns}")
            continue
        base = subs[RETRIEVERS[0]][REF_COL]
        for r in RETRIEVERS[1:]:
            mism = (subs[r][REF_COL] != base)
            if mism.any():
                raise AssertionError(
                    f"{task}: {int(mism.sum())} reference mismatches between "
                    f"'{RETRIEVERS[0]}' and '{r}' — row order differs.")
        refs = base.tolist()
        # exclude instances with empty gold reference
        keep = [i for i, ref in enumerate(refs) if ref.strip()]
        if len(keep) < len(refs):
            print(f"  [WARN] {task}: {len(refs)-len(keep)} empty gold refs excluded")
        per_task[task] = ([refs[i] for i in keep],
                          {r: [subs[r][TEXT_COL][i] for i in keep]
                           for r in RETRIEVERS},
                          keep)  # keep = original instance_idx values
    return per_task

# ------------------------------------------------------------------------
# Score
# ------------------------------------------------------------------------

from bert_score import score as bert_score

per_output_rows = []
data = load_retriever_outputs()

for task, (refs, outs_by_retr, idxs) in data.items():
    print(f"\n=== {task}: {len(refs)} instances x {len(RETRIEVERS)} retrievers ===")
    for retr in RETRIEVERS:
        outs = outs_by_retr[retr]
        cands, crefs, valid_idx, empty_idx = [], [], [], []
        for iid, out, ref in zip(idxs, outs, refs):
            if out and out.strip():
                cands.append(out); crefs.append(ref); valid_idx.append(iid)
            else:
                empty_idx.append(iid)

        if cands:
            _, _, F = bert_score(cands, crefs, model_type=BERT_MODEL,
                                 lang='en', rescale_with_baseline=BERT_RESCALE,
                                 batch_size=BERT_BATCH, device=DEVICE, verbose=False)
            F = F.numpy()
        else:
            F = np.array([])

        for iid, f1 in zip(valid_idx, F):
            per_output_rows.append({
                'model': MODEL_NAME, 'task': task, 'retriever': retr,
                'instance_idx': iid, 'is_empty': False,
                'bertscore_f1': float(f1),
            })
        for iid in empty_idx:
            per_output_rows.append({
                'model': MODEL_NAME, 'task': task, 'retriever': retr,
                'instance_idx': iid, 'is_empty': True,
                'bertscore_f1': np.nan,
            })
        print(f"  [done] {retr}: {len(cands)} scored, {len(empty_idx)} empty")

if not per_output_rows:
    raise SystemExit("[ERROR] nothing scored — check TASK_MAP against the "
                     "'dataset' values in the per-example CSVs.")

# ------------------------------------------------------------------------
# Aggregate + save
# ------------------------------------------------------------------------

po = pd.DataFrame(per_output_rows)
po.to_csv(os.path.join(OUT_DIR, f's1dynbs_{MODEL_SLUG}_per_output.csv'), index=False)

# per instance, wide: one column per retriever + cross-retriever stats
wide = po.pivot_table(index=['task', 'instance_idx'],
                      columns='retriever', values='bertscore_f1')
wide.columns = [f'{r}_bertscore' for r in wide.columns]
bs_cols = [f'{r}_bertscore' for r in RETRIEVERS]
wide['n_valid_retr_outputs'] = wide[bs_cols].notna().sum(axis=1)
wide['retr_mean_bertscore'] = wide[bs_cols].mean(axis=1)
wide['retr_min_bertscore'] = wide[bs_cols].min(axis=1)
wide['retr_max_bertscore'] = wide[bs_cols].max(axis=1)
wide['retr_bertscore_spread'] = wide.retr_max_bertscore - wide.retr_min_bertscore
pi = wide.reset_index()
pi.insert(0, 'model', MODEL_NAME)
per_inst_csv = os.path.join(OUT_DIR, f's1dynbs_{MODEL_SLUG}_per_instance.csv')
pi.to_csv(per_inst_csv, index=False)

summary = (po.groupby(['task', 'retriever'])
             .agg(n_instances=('instance_idx', 'count'),
                  n_empty=('is_empty', 'sum'),
                  mean_bertscore=('bertscore_f1', 'mean'),
                  median_bertscore=('bertscore_f1', 'median'))
             .round(4)
             .reset_index())
summary.insert(0, 'model', MODEL_NAME)
summary['bert_model'] = BERT_MODEL
summary['bert_rescaled'] = BERT_RESCALE
summary_csv = os.path.join(OUT_DIR, f's1dynbs_{MODEL_SLUG}_summary.csv')
summary.to_csv(summary_csv, index=False)

print(f"\n### S1-dynamic BERTScore summary — {MODEL_NAME} ###")
print(summary.to_string(index=False))

# best retriever per task by mean_bertscore (appendix BERTScore version of T1 'Dyn')
best = summary.loc[summary.groupby('task')['mean_bertscore'].idxmax(),
                   ['task', 'retriever', 'mean_bertscore']]
print("\nBest retriever per task (by BERTScore):")
print(best.to_string(index=False))

print(f"\nSaved -> {per_inst_csv}")
print(f"      -> {summary_csv}")
print("\nE3/E5 join: merge with s3retr_..._per_instance.csv on "
      "(task, instance_idx).")


Per-example CSVs:
  s_pubmedbert: results_dynamic_k5_pool100_medalpaca_7b_s_pubmedbert_ms_marco_per_example.csv
  bge: results_dynamic_k5_pool100_medalpaca_7b_bge_base_en_v1_5_per_example.csv
  tfidf: results_dynamic_k5_pool100_medalpaca_7b_tfidf_per_example.csv
  [WARN] medicationqa: 1 empty gold refs excluded
  [WARN] mtsamples_proc: 1 empty gold refs excluded

=== meddialog: 500 instances x 3 retrievers ===
  [done] s_pubmedbert: 500 scored, 0 empty
  [done] bge: 500 scored, 0 empty
  [done] tfidf: 500 scored, 0 empty

=== medicationqa: 499 instances x 3 retrievers ===
  [done] s_pubmedbert: 499 scored, 0 empty
  [done] bge: 499 scored, 0 empty
  [done] tfidf: 499 scored, 0 empty

=== mtsamples: 500 instances x 3 retrievers ===
  [done] s_pubmedbert: 491 scored, 9 empty
  [done] bge: 490 scored, 10 empty
  [done] tfidf: 494 scored, 6 empty

=== mtsamples_proc: 499 instances x 3 retrievers ===
  [done] s_pubmedbert: 493 scored, 6 empty
  [done] bge: 494 scored, 5 empty
  [done] tfidf